# Cross-Attention Transformer: Peptide (ESM-2) × Bacteria (Oligonucleotide) → MIC
**Google Colab notebook** — clones the project from GitHub and reads data directly from the repo.

### Architecture
- **Peptide branch**: ESM-2 embedding (480d) → projection → tokens → self-attention
- **Bacteria branch**: Oligonucleotide composition k=3,4,5 (1344d) → projection → tokens → self-attention
- **Cross-attention**: bidirectional peptide ↔ bacteria interaction (4 layers)
- **Fusion**: CLS token pooling → MLP regression head


In [16]:
#@title 1. Clone repo & install dependencies
import os

# Clone the project
REPO_URL = "https://github.com/nmach22/amp-prediction.git"
PROJECT_DIR = "/content/amp-prediction"

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
os.chdir(PROJECT_DIR)

# Install dependencies
!pip -q install "transformers>=4.40" pyarrow scikit-learn scipy seaborn 2>/dev/null

import torch, transformers
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"transformers {transformers.__version__}")
print(f"Working dir: {os.getcwd()}")


torch 2.11.0+cu128 | CUDA: True
GPU: Tesla T4
transformers 5.13.1
Working dir: /content/amp-prediction


In [17]:
#@title 2. Verify data files exist in repo
from pathlib import Path

required_files = {
    "data/raw/amp_mic_activities.csv": "Raw MIC data",
    "data/processed/embeddings/genome/oligo_composition.parquet": "Bacteria genome vectors",
}

for fpath, desc in required_files.items():
    exists = Path(fpath).exists()
    status = "✓" if exists else "✗ MISSING"
    print(f"  {status} {desc}: {fpath}")
    if not exists:
        raise FileNotFoundError(f"Required file missing: {fpath}")

# Check optional ESM-2 cache
esm_cache = Path("data/processed/embeddings/facebook_esm2_t12_35M_UR50D_mic_embeddings.npz")
HAS_ESM_CACHE = esm_cache.exists()
print(f"{"✓" if HAS_ESM_CACHE else "○"} ESM-2 cache: {"available (will skip computation)" if HAS_ESM_CACHE else "not found (will compute on GPU)"}")


  ✓ Raw MIC data: data/raw/amp_mic_activities.csv
  ✓ Bacteria genome vectors: data/processed/embeddings/genome/oligo_composition.parquet
✓ ESM-2 cache: available (will skip computation)


In [18]:
#@title 3. Load & prepare data
import sys
sys.path.insert(0, ".")

import numpy as np
import pandas as pd
import re

# Load raw MIC data
df = pd.read_csv("data/raw/amp_mic_activities.csv")
print(f"Raw rows: {len(df):,}")

# Clean
df = df.dropna(subset=["sequence", "target_activity_name", "activity"])
df["sequence"] = df["sequence"].str.upper().str.strip()
df["target_activity_name"] = df["target_activity_name"].str.strip()
df["activity"] = pd.to_numeric(df["activity"], errors="coerce")
df = df[df["activity"] > 0]
df = df[df["sequence"].str.len() > 0]
df = df[~df["sequence"].str.contains(r"[^ACDEFGHIKLMNPQRSTVWY]", flags=re.IGNORECASE)]
df["log_mic"] = np.log10(df["activity"])
df["species"] = df["target_activity_name"].apply(lambda x: " ".join(str(x).split()[:2]))

# Aggregate replicates (geometric mean in log space)
agg = df.groupby(["sequence", "species"], as_index=False).agg(
    log_mic=("log_mic", "mean"),
    n_measurements=("log_mic", "size"),
    target_activity_name=("target_activity_name", "first")
)
print(f"After aggregation: {len(agg):,} unique (peptide, species) pairs")
print(f"Unique peptides: {agg["sequence"].nunique():,}")
print(f"Unique species: {agg["species"].nunique():,}")
print(f"log₁₀(MIC) stats: {agg["log_mic"].describe()}")


Raw rows: 114,880
After aggregation: 66,894 unique (peptide, species) pairs
Unique peptides: 13,317
Unique species: 718
log₁₀(MIC) stats: count    66894.000000
mean         1.357825
std          0.792260
min         -4.840887
25%          0.828379
50%          1.384872
75%          1.933513
max          4.836751
Name: log_mic, dtype: float64


In [19]:
#@title 4. Load oligonucleotide genome features
from src.features.genome import GenomeEncoder

# Use project's GenomeEncoder to load oligo features
genome_enc = GenomeEncoder()
oligo = pd.read_parquet("data/processed/embeddings/genome/oligo_composition.parquet")
print(f"Oligo features: {oligo.shape[0]} species × {oligo.shape[1]-1} features")

# Merge with data
oligo_cols = [c for c in oligo.columns if c != "species"]
agg = agg.merge(oligo, on="species", how="left")

# Fill missing genome features with zeros
matched = agg[oligo_cols[0]].notna().sum()
print(f"Genome match rate: {matched}/{len(agg)} ({matched/len(agg)*100:.1f}%)")
agg[oligo_cols] = agg[oligo_cols].fillna(0.0)


Oligo features: 648 species × 1344 features
Genome match rate: 65898/66894 (98.5%)


In [20]:
#@title 5. Load or compute ESM-2 peptide embeddings
from pathlib import Path

ESM2_MODEL = "facebook/esm2_t12_35M_UR50D"
ESM2_DIM = 480
CACHE_PATH = Path("data/processed/embeddings/facebook_esm2_t12_35M_UR50D_mic_embeddings.npz")

unique_seqs = sorted(agg["sequence"].unique())
print(f"Need embeddings for {len(unique_seqs)} unique sequences")

# Try loading from cache
seq_to_emb = {}
if CACHE_PATH.exists():
    cache = np.load(CACHE_PATH, allow_pickle=True)
    cached_seqs = cache["sequences"]
    cached_embs = cache["embeddings"]
    for s, e in zip(cached_seqs, cached_embs):
        seq_to_emb[s] = e
    print(f"Loaded {len(seq_to_emb)} embeddings from cache")

# Find missing sequences
missing_seqs = [s for s in unique_seqs if s not in seq_to_emb]
print(f"Missing sequences: {len(missing_seqs)}")

if missing_seqs:
    from transformers import AutoTokenizer, AutoModel
    print(f"Computing ESM-2 embeddings for {len(missing_seqs)} sequences on {DEVICE}...")

    tokenizer = AutoTokenizer.from_pretrained(ESM2_MODEL)
    esm_model = AutoModel.from_pretrained(ESM2_MODEL).eval().to(DEVICE)

    batch_size = 32
    with torch.no_grad():
        for i in range(0, len(missing_seqs), batch_size):
            batch = missing_seqs[i:i+batch_size]
            inputs = tokenizer(
                batch, return_tensors="pt", padding=True,
                truncation=True, max_length=512,
                return_special_tokens_mask=True
            ).to(DEVICE)

            outputs = esm_model(**inputs)
            mask = inputs["attention_mask"].bool() & ~inputs["special_tokens_mask"].bool()
            mask_3d = mask.unsqueeze(-1).float()
            pooled = (outputs.last_hidden_state * mask_3d).sum(1) / mask_3d.sum(1).clamp(min=1)
            embs = pooled.cpu().numpy()

            for seq, emb in zip(batch, embs):
                seq_to_emb[seq] = emb

            if (i // batch_size) % 50 == 0:
                print(f"  [{i+len(batch)}/{len(missing_seqs)}]")

    del esm_model
    torch.cuda.empty_cache()

    # Save updated cache
    all_seqs = sorted(seq_to_emb.keys())
    all_embs = np.array([seq_to_emb[s] for s in all_seqs])
    np.savez(CACHE_PATH, sequences=all_seqs, embeddings=all_embs)
    print(f"Updated cache: {len(all_seqs)} sequences → {CACHE_PATH}")

# Build embedding matrix aligned with agg
peptide_embeddings = np.vstack([seq_to_emb[s] for s in agg["sequence"]])
print(f"✓ Peptide embeddings: {peptide_embeddings.shape}")


Need embeddings for 13317 unique sequences
Loaded 13317 embeddings from cache
Missing sequences: 0
✓ Peptide embeddings: (66894, 480)


In [21]:
#@title 6. Train/validation split (grouped by peptide to avoid leakage)
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_idx, val_idx = next(gss.split(agg, groups=agg["sequence"]))

print(f"Train: {len(train_idx):,} | Val: {len(val_idx):,}")
print(f"Train peptides: {agg.iloc[train_idx]["sequence"].nunique()}")
print(f"Val peptides: {agg.iloc[val_idx]["sequence"].nunique()}")

# Prepare tensors
genome_features = agg[oligo_cols].values.astype(np.float32)

X_pep_train = peptide_embeddings[train_idx]
X_pep_val = peptide_embeddings[val_idx]
X_gen_train = genome_features[train_idx]
X_gen_val = genome_features[val_idx]
y_train = agg["log_mic"].values[train_idx].astype(np.float32)
y_val = agg["log_mic"].values[val_idx].astype(np.float32)

print(f"Feature dims: peptide={X_pep_train.shape[1]}, genome={X_gen_train.shape[1]}")


Train: 56,924 | Val: 9,970
Train peptides: 11319
Val peptides: 1998
Feature dims: peptide=480, genome=1344


In [22]:
#@title 7. Normalize features
from sklearn.preprocessing import StandardScaler

pep_scaler = StandardScaler()
gen_scaler = StandardScaler()

X_pep_train = pep_scaler.fit_transform(X_pep_train)
X_pep_val = pep_scaler.transform(X_pep_val)
X_gen_train = gen_scaler.fit_transform(X_gen_train)
X_gen_val = gen_scaler.transform(X_gen_val)

print(f"Peptide features: mean={X_pep_train.mean():.4f}, std={X_pep_train.std():.4f}")
print(f"Genome features: mean={X_gen_train.mean():.4f}, std={X_gen_train.std():.4f}")


Peptide features: mean=0.0000, std=1.0000
Genome features: mean=0.0000, std=1.0000


In [23]:
#@title 8. Define Deeper Cross-Attention Transformer
# Reuse building blocks from the project, but define a deeper variant here
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


class StochasticDepth(nn.Module):
    """Drop entire layer with probability p during training."""
    def __init__(self, p: float = 0.1):
        super().__init__()
        self.p = p

    def forward(self, x: torch.Tensor, residual: torch.Tensor) -> torch.Tensor:
        if not self.training or self.p == 0:
            return x + residual
        mask = torch.bernoulli(torch.ones(1, device=x.device) * (1 - self.p))
        return x + residual * mask / (1 - self.p)


class SelfAttentionBlock(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1, drop_path=0.0):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model),
            nn.Dropout(dropout),
        )
        self.norm2 = nn.LayerNorm(d_model)
        self.drop_path = StochasticDepth(drop_path)

    def forward(self, x):
        attn_out, _ = self.self_attn(x, x, x)
        x = self.drop_path(x, self.norm1(attn_out))
        x = self.drop_path(x, self.norm2(self.ffn(x)))
        return x


class CrossAttentionBlock(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1, drop_path=0.0):
        super().__init__()
        self.cross_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model),
            nn.Dropout(dropout),
        )
        self.norm2 = nn.LayerNorm(d_model)
        self.drop_path = StochasticDepth(drop_path)

    def forward(self, query, context):
        attn_out, _ = self.cross_attn(query, context, context)
        x = self.drop_path(query, self.norm1(attn_out))
        x = self.drop_path(x, self.norm2(self.ffn(x)))
        return x


class DeepCrossAttentionTransformer(nn.Module):
    """
    Deeper cross-attention transformer for peptide × bacteria MIC prediction.
    d_model=384, 12 heads, 3 self + 4 cross layers, stochastic depth.
    """

    def __init__(
        self,
        peptide_dim=480,
        genome_dim=1344,
        d_model=384,
        n_heads=12,
        n_self_layers=3,
        n_cross_layers=4,
        n_tokens_peptide=10,
        n_tokens_genome=21,
        dropout=0.2,
        drop_path_rate=0.05,
    ):
        super().__init__()
        self.d_model = d_model
        self.n_tokens_peptide = n_tokens_peptide
        self.n_tokens_genome = n_tokens_genome

        # Projection: raw features → token sequence
        self.pep_proj = nn.Sequential(
            nn.Linear(peptide_dim, d_model * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 2, n_tokens_peptide * d_model),
        )
        self.gen_proj = nn.Sequential(
            nn.Linear(genome_dim, d_model * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 2, n_tokens_genome * d_model),
        )

        # Learnable CLS tokens
        self.cls_pep = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.cls_gen = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)

        # Learnable positional embeddings
        self.pos_pep = nn.Parameter(torch.randn(1, n_tokens_peptide + 1, d_model) * 0.02)
        self.pos_gen = nn.Parameter(torch.randn(1, n_tokens_genome + 1, d_model) * 0.02)

        # Stochastic depth schedule
        dp_rates_self = [drop_path_rate * i / (n_self_layers - 1) for i in range(n_self_layers)] if n_self_layers > 1 else [0.0]
        dp_rates_cross = [drop_path_rate * i / (n_cross_layers - 1) for i in range(n_cross_layers)] if n_cross_layers > 1 else [0.0]

        # Self-attention blocks
        self.self_pep = nn.ModuleList([
            SelfAttentionBlock(d_model, n_heads, dropout, dp_rates_self[i])
            for i in range(n_self_layers)
        ])
        self.self_gen = nn.ModuleList([
            SelfAttentionBlock(d_model, n_heads, dropout, dp_rates_self[i])
            for i in range(n_self_layers)
        ])

        # Cross-attention blocks (bidirectional)
        self.cross_p2g = nn.ModuleList([
            CrossAttentionBlock(d_model, n_heads, dropout, dp_rates_cross[i])
            for i in range(n_cross_layers)
        ])
        self.cross_g2p = nn.ModuleList([
            CrossAttentionBlock(d_model, n_heads, dropout, dp_rates_cross[i])
            for i in range(n_cross_layers)
        ])

        # Final norms
        self.norm_pep = nn.LayerNorm(d_model)
        self.norm_gen = nn.LayerNorm(d_model)

        # Regression head
        self.head = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(d_model // 2, 1),
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, peptide_features, genome_features):
        B = peptide_features.size(0)

        # Project to token sequences
        pep = self.pep_proj(peptide_features).view(B, self.n_tokens_peptide, self.d_model)
        gen = self.gen_proj(genome_features).view(B, self.n_tokens_genome, self.d_model)

        # Prepend CLS + add positional
        pep = torch.cat([self.cls_pep.expand(B, -1, -1), pep], dim=1) + self.pos_pep
        gen = torch.cat([self.cls_gen.expand(B, -1, -1), gen], dim=1) + self.pos_gen

        # Self-attention
        for layer in self.self_pep:
            pep = layer(pep)
        for layer in self.self_gen:
            gen = layer(gen)

        # Cross-attention (bidirectional)
        for p2g, g2p in zip(self.cross_p2g, self.cross_g2p):
            pep_new = p2g(pep, gen)
            gen_new = g2p(gen, pep)
            pep, gen = pep_new, gen_new

        # Pool CLS tokens
        pep_cls = self.norm_pep(pep[:, 0])
        gen_cls = self.norm_gen(gen[:, 0])

        # Fuse and predict
        fused = torch.cat([pep_cls, gen_cls], dim=1)
        return self.head(fused)


# Instantiate
model = DeepCrossAttentionTransformer(
    peptide_dim=X_pep_train.shape[1],
    genome_dim=X_gen_train.shape[1],
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")
print(f"Architecture: d_model=384, heads=12, self_layers=3, cross_layers=4")
print(f"Input: peptide({X_pep_train.shape[1]}d) + genome({X_gen_train.shape[1]}d)")


Model parameters: 35,783,425
Architecture: d_model=384, heads=12, self_layers=3, cross_layers=4
Input: peptide(480d) + genome(1344d)


In [24]:
#@title 9. PyTorch Dataset with Mixup augmentation

class MICDataset(torch.utils.data.Dataset):
    def __init__(self, pep, gen, targets):
        self.pep = torch.tensor(pep, dtype=torch.float32)
        self.gen = torch.tensor(gen, dtype=torch.float32)
        self.targets = torch.tensor(targets, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        return self.pep[idx], self.gen[idx], self.targets[idx]


def mixup_batch(pep, gen, y, alpha=0.2):
    """Mixup augmentation — interpolate random pairs."""
    if alpha <= 0:
        return pep, gen, y
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(pep.size(0), device=pep.device)
    pep_mix = lam * pep + (1 - lam) * pep[idx]
    gen_mix = lam * gen + (1 - lam) * gen[idx]
    y_mix = lam * y + (1 - lam) * y[idx]
    return pep_mix, gen_mix, y_mix


train_dataset = MICDataset(X_pep_train, X_gen_train, y_train)
val_dataset = MICDataset(X_pep_val, X_gen_val, y_val)

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=256, shuffle=True, num_workers=2, pin_memory=True
)
val_loader = torch.utils.data.DataLoader(
    val_dataset, batch_size=512, shuffle=False, num_workers=2, pin_memory=True
)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")


Train batches: 223 | Val batches: 20


In [25]:
#@title 10. Training loop with cosine annealing + warm restarts
import time
from collections import defaultdict

# Hyperparameters
MAX_EPOCHS = 500
PATIENCE = 50
LR = 3e-4
WEIGHT_DECAY = 5e-4
GRAD_ACCUM_STEPS = 2  # effective batch = 256 * 2 = 512
MIXUP_ALPHA = 0.1

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=50, T_mult=2, eta_min=1e-6
)
loss_fn = nn.HuberLoss(delta=1.0)

history = defaultdict(list)
best_val_mae = float("inf")
best_state = None
epochs_no_improve = 0

print(f"Training for up to {MAX_EPOCHS} epochs (patience={PATIENCE})")
print(f"Effective batch size: {256 * GRAD_ACCUM_STEPS} | Mixup α={MIXUP_ALPHA}")
print("-" * 70)

for epoch in range(1, MAX_EPOCHS + 1):
    t0 = time.time()

    # ─── Train ───
    model.train()
    train_losses = []
    optimizer.zero_grad()

    for step, (pep, gen, y) in enumerate(train_loader):
        pep, gen, y = pep.to(DEVICE), gen.to(DEVICE), y.to(DEVICE)
        pep, gen, y = mixup_batch(pep, gen, y, alpha=MIXUP_ALPHA)

        pred = model(pep, gen)
        loss = loss_fn(pred, y) / GRAD_ACCUM_STEPS
        loss.backward()

        if (step + 1) % GRAD_ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()

        train_losses.append(loss.item() * GRAD_ACCUM_STEPS)

    scheduler.step()

    # ─── Validate ───
    model.eval()
    val_preds, val_targets = [], []
    with torch.no_grad():
        for pep, gen, y in val_loader:
            pep, gen = pep.to(DEVICE), gen.to(DEVICE)
            pred = model(pep, gen).cpu()
            val_preds.append(pred)
            val_targets.append(y)

    val_preds = torch.cat(val_preds).numpy().ravel()
    val_targets = torch.cat(val_targets).numpy().ravel()
    val_mae = np.mean(np.abs(val_preds - val_targets))
    val_rmse = np.sqrt(np.mean((val_preds - val_targets)**2))
    ss_res = np.sum((val_targets - val_preds)**2)
    ss_tot = np.sum((val_targets - val_targets.mean())**2)
    val_r2 = 1 - ss_res / ss_tot

    # Train MAE (collect targets from loader to match shuffle order)
    train_preds, train_targets = [], []
    with torch.no_grad():
        for pep, gen, y in train_loader:
            pep, gen = pep.to(DEVICE), gen.to(DEVICE)
            train_preds.append(model(pep, gen).cpu())
            train_targets.append(y)
    train_preds = torch.cat(train_preds).numpy().ravel()
    train_targets = torch.cat(train_targets).numpy().ravel()
    train_mae = np.mean(np.abs(train_preds - train_targets))

    # Log
    elapsed = time.time() - t0
    history["train_mae"].append(train_mae)
    history["val_mae"].append(val_mae)
    history["val_rmse"].append(val_rmse)
    history["val_r2"].append(val_r2)
    history["lr"].append(optimizer.param_groups[0]["lr"])

    # Early stopping
    if val_mae < best_val_mae - 1e-4:
        best_val_mae = val_mae
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        epochs_no_improve = 0
        marker = " ★"
    else:
        epochs_no_improve += 1
        marker = ""

    if epoch % 5 == 0 or epochs_no_improve == 0:
        print(f"Epoch {epoch:3d} | train_mae={train_mae:.4f} | "
              f"val_mae={val_mae:.4f} | val_r²={val_r2:.4f} | "
              f"lr={optimizer.param_groups[0]["lr"]:.2e} | {elapsed:.1f}s{marker}")

    if epochs_no_improve >= PATIENCE:
        print(f"✗ Early stopping at epoch {epoch} (best val_mae={best_val_mae:.4f})")
        break

# Restore best
if best_state:
    model.load_state_dict(best_state)
    model.to(DEVICE)
print(f"✓ Best model: val_mae={best_val_mae:.4f}")


Training for up to 500 epochs (patience=50)
Effective batch size: 512 | Mixup α=0.1
----------------------------------------------------------------------
Epoch   1 | train_mae=0.6977 | val_mae=0.5775 | val_r²=0.1877 | lr=3.00e-04 | 89.1s ★
Epoch   2 | train_mae=0.7210 | val_mae=0.5265 | val_r²=0.2882 | lr=2.99e-04 | 86.3s ★
Epoch   4 | train_mae=0.7320 | val_mae=0.4985 | val_r²=0.3479 | lr=2.95e-04 | 86.7s ★
Epoch   5 | train_mae=0.7132 | val_mae=0.5138 | val_r²=0.3170 | lr=2.93e-04 | 86.0s


KeyboardInterrupt: 

In [ ]:
#@title 11. Final evaluation & plots
import matplotlib.pyplot as plt
import seaborn as sns

# Final predictions
model.eval()
with torch.no_grad():
    final_preds = []
    for pep, gen, y in val_loader:
        pep, gen = pep.to(DEVICE), gen.to(DEVICE)
        final_preds.append(model(pep, gen).cpu())
    final_preds = torch.cat(final_preds).numpy().ravel()

final_mae = np.mean(np.abs(final_preds - y_val))
final_rmse = np.sqrt(np.mean((final_preds - y_val)**2))
ss_res = np.sum((y_val - final_preds)**2)
ss_tot = np.sum((y_val - y_val.mean())**2)
final_r2 = 1 - ss_res / ss_tot

print(f"{"="*50}")
print(f"FINAL VALIDATION METRICS")
print(f"{"="*50}")
print(f"  MAE:  {final_mae:.4f}")
print(f"  RMSE: {final_rmse:.4f}")
print(f"  R²:   {final_r2:.4f}")
print(f"{"="*50}")

# ─── Plots ───
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Learning curves
ax = axes[0]
ax.plot(history["train_mae"], label="Train MAE", alpha=0.8)
ax.plot(history["val_mae"], label="Val MAE", alpha=0.8)
ax.set_xlabel("Epoch")
ax.set_ylabel("MAE")
ax.set_title("Learning Curves")
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Predicted vs Actual
ax = axes[1]
ax.scatter(y_val, final_preds, alpha=0.1, s=5)
lims = [min(y_val.min(), final_preds.min()), max(y_val.max(), final_preds.max())]
ax.plot(lims, lims, "r--", linewidth=1)
ax.set_xlabel("Actual log₁₀(MIC)")
ax.set_ylabel("Predicted log₁₀(MIC)")
ax.set_title(f"Predicted vs Actual (R²={final_r2:.3f})")
ax.grid(True, alpha=0.3)

# 3. Residual distribution
ax = axes[2]
residuals = final_preds - y_val
ax.hist(residuals, bins=50, edgecolor="white", alpha=0.7)
ax.axvline(0, color="red", linestyle="--")
ax.set_xlabel("Residual (pred - actual)")
ax.set_ylabel("Count")
ax.set_title(f"Residuals (MAE={final_mae:.3f})")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("results/figures/genome_transformer_v2.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
#@title 12. Save model checkpoint
from pathlib import Path

Path("results/checkpoints").mkdir(parents=True, exist_ok=True)

checkpoint = {
    "model_state_dict": model.state_dict(),
    "pep_scaler_mean": pep_scaler.mean_,
    "pep_scaler_scale": pep_scaler.scale_,
    "gen_scaler_mean": gen_scaler.mean_,
    "gen_scaler_scale": gen_scaler.scale_,
    "config": {
        "peptide_dim": X_pep_train.shape[1],
        "genome_dim": X_gen_train.shape[1],
        "d_model": 384,
        "n_heads": 12,
        "n_self_layers": 3,
        "n_cross_layers": 4,
        "n_tokens_peptide": 10,
        "n_tokens_genome": 21,
        "dropout": 0.2,
    },
    "metrics": {
        "best_val_mae": best_val_mae,
        "final_mae": final_mae,
        "final_rmse": final_rmse,
        "final_r2": final_r2,
    },
    "history": dict(history),
}

ckpt_path = "results/checkpoints/genome_transformer_mic_v2.pt"
torch.save(checkpoint, ckpt_path)
print(f"✓ Saved: {ckpt_path}")

# Download from Colab
from google.colab import files
files.download(ckpt_path)
files.download("results/figures/genome_transformer_v2.png")
